# Stage 00 v2 — Raw data preparation (ingestion delgada)

```
S00_raw_data_preparation_v2.ipynb
```

## Alcance

Esta notebook **no contiene logica de negocio**. Toda la logica de
ingestion, validacion, deteccion de gaps, manifiesto y escritura atomica
vive en `src/data/s00_raw_ingestion.py`. Esta notebook solo la invoca y
muestra los resultados.

- No modifica `notebooks/S00_raw_data_preparation.ipynb` (permanece intacta
  como evidencia historica).
- No modifica los archivos fuente en `data/00_source/`.
- No aplica ninguna transformacion temporal (sin `tz_localize`, sin
  `tz_convert`, sin filtrado de sesion, sin calendario) -- eso pertenece a
  S01.
- No descarta, corrige ni rellena filas invalidas: cualquier fila que
  falle una validacion detiene la generacion del artefacto.
- Los artefactos generados usan el sufijo `_v2` y no tocan los nombres
  oficiales anteriores (`mnq_raw.parquet`, `mnq_raw_summary.json`).

## 0. Configuracion del entorno

In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path, marker: str = "config/data_config.yaml", max_levels: int = 15) -> Path:
    """Busca hacia arriba un directorio que contenga `marker`, con limite de
    profundidad explicito -- a diferencia de la notebook original, no depende
    de que la carpeta se llame literalmente 'neural_profit' ni puede quedar
    en bucle infinito si el marcador no existe."""
    current = start.resolve()
    for _ in range(max_levels):
        if (current / marker).exists():
            return current
        if current.parent == current:
            break
        current = current.parent
    raise FileNotFoundError(
        f"No se encontro un ancestro de {start} que contenga {marker} en {max_levels} niveles"
    )


PROJECT_ROOT = find_project_root(Path.cwd())
print("Project root:", PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

Project root: C:\Users\heguu\OneDrive\Escritorio\neural_profit


In [2]:
from src.data import s00_raw_ingestion as ing

CONFIG_PATH = PROJECT_ROOT / "config" / "data_config.yaml"
print("Config:", CONFIG_PATH)

Config: C:\Users\heguu\OneDrive\Escritorio\neural_profit\config\data_config.yaml


## 1. Ejecucion productiva

Genera (o reutiliza, si nada cambio segun la regla de staleness de
`config/data_config.yaml` + hashes de codigo/fuentes) los artefactos
oficiales de S00 v2 en `data/01_raw/`.

In [3]:
result = ing.run_s00_ingestion(project_root=PROJECT_ROOT, config_path=CONFIG_PATH)

print("Reutilizo artefacto existente:", result.reused_existing)
print("Parquet:", result.parquet_path)
print("SHA-256 Parquet:", result.parquet_sha256)
print("Manifest:", result.manifest_path)
print("Summary:", result.summary_path)
print("Gaps:", result.gaps_path)
print("Source manifest CSV (derivado):", result.source_manifest_csv_path)

Reutilizo artefacto existente: False
Parquet: C:\Users\heguu\OneDrive\Escritorio\neural_profit\data\01_raw\mnq_raw_v2.parquet
SHA-256 Parquet: dfe1490ac9e9a0c85c4e2ae57c8fbbb90e84a5ee369146941f28e5e047c5beb0
Manifest: C:\Users\heguu\OneDrive\Escritorio\neural_profit\data\01_raw\mnq_raw_v2_manifest.json
Summary: C:\Users\heguu\OneDrive\Escritorio\neural_profit\data\01_raw\mnq_raw_v2_summary.json
Gaps: C:\Users\heguu\OneDrive\Escritorio\neural_profit\data\01_raw\mnq_raw_v2_gaps.parquet
Source manifest CSV (derivado): C:\Users\heguu\OneDrive\Escritorio\neural_profit\manifests\s00_source_manifest.csv


## 2. Resultado del dataset consolidado

In [4]:
df = result.df
print("Shape:", df.shape)
print("Columnas:", list(df.columns))
print("Rango:", df.index.min(), "->", df.index.max())
print("Index tz:", df.index.tz)
df.head()

Shape: (2172640, 6)
Columnas: ['open', 'high', 'low', 'close', 'volume', 'contract']
Rango: 2019-12-23 03:01:00 -> 2026-04-17 20:18:00
Index tz: None


,open,high,low,close,volume,contract
datetime,,,,,,
2019-12-23 03:01:00,8718.50,8718.75,8718.50,8718.50,9,H20
2019-12-23 03:02:00,8718.25,8718.25,8718.00,8718.25,14,H20
2019-12-23 03:03:00,8718.25,8718.50,8718.00,8718.25,74,H20
2019-12-23 03:04:00,8718.25,8719.00,8718.25,8718.50,10,H20
2019-12-23 03:05:00,8718.50,8719.00,8718.50,8719.00,6,H20


In [5]:
df.tail()

,open,high,low,close,volume,contract
datetime,,,,,,
2026-04-17 20:14:00,26835.50,26844.50,26835.50,26839.25,1120,M26
2026-04-17 20:15:00,26839.50,26842.25,26837.75,26840.25,605,M26
2026-04-17 20:16:00,26840.75,26841.75,26835.75,26841.25,615,M26
2026-04-17 20:17:00,26840.50,26844.75,26840.00,26842.25,410,M26
2026-04-17 20:18:00,26842.50,26843.75,26839.50,26840.25,221,M26


## 3. Validaciones ejecutadas

Si esta celda se ejecuto sin excepciones, significa que sobre los 26
archivos fuente se verificaron -- sin descartar ninguna fila -- schema,
parseo, timestamps, monotonicidad, duplicados globales, duplicados por
(timestamp, contract), nulos, infinitos, precios positivos, volumen no
negativo, volumen entero, invariantes OHLC, filas exactamente duplicadas,
archivos vacios y transiciones de contrato. Ver
`src/data/s00_raw_ingestion.py` para el detalle de cada chequeo.

In [6]:
print("Archivos fuente inventariados:", len(result.source_records))
for r in result.source_records:
    print(f"  {r['filename']:24s} {r['instrument']}{'':1s} contract={r['contract']:4s} "
          f"contract_full={r['contract_full']:7s} n_rows={r['n_rows']:6d} "
          f"[{r['first_timestamp']} -> {r['last_timestamp']}]")

Archivos fuente inventariados: 26
  00_mnq_03_20.Last.txt    MNQ  contract=H20  contract_full=MNQH20  n_rows= 81660 [2019-12-23T03:01:00 -> 2020-03-20T13:30:00]
  01_mnq_06_20.Last.txt    MNQ  contract=M20  contract_full=MNQM20  n_rows= 86069 [2020-03-23T03:01:00 -> 2020-06-19T13:30:00]
  02_mnq_09_20.Last.txt    MNQ  contract=U20  contract_full=MNQU20  n_rows= 87482 [2020-06-22T03:01:00 -> 2020-09-18T13:30:00]
  03_mnq_12_20.Last.txt    MNQ  contract=Z20  contract_full=MNQZ20  n_rows= 86824 [2020-09-21T03:01:00 -> 2020-12-18T03:00:00]
  04_mnq_03_21.Last.txt    MNQ  contract=H21  contract_full=MNQH21  n_rows= 84599 [2020-12-21T03:01:00 -> 2021-03-19T13:30:00]
  05_mnq_06_21.Last.txt    MNQ  contract=M21  contract_full=MNQM21  n_rows= 87090 [2021-03-22T03:01:00 -> 2021-06-18T13:30:00]
  06_mnq_09_21.Last.txt    MNQ  contract=U21  contract_full=MNQU21  n_rows= 88205 [2021-06-21T03:01:00 -> 2021-09-17T13:30:00]
  07_mnq_12_21.Last.txt    MNQ  contract=Z21  contract_full=MNQZ21  n_rows= 8

## 4. Catalogo de gaps

El detalle fila a fila vive en `mnq_raw_v2_gaps.parquet` (no en el
manifest/summary, que solo contienen agregaciones). Ningun gap se clasifica
aqui como feriado/mantenimiento/jornada de trading de forma definitiva --
eso requiere el calendario y la zona horaria confirmada que S01 todavia no
tiene.

In [7]:
print("Total de gaps registrados:", result.manifest["gaps"]["total_gaps"])
print("Por bucket estructural:", result.manifest["gaps"]["by_structural_bucket"])
print("Por evidence_level:", result.manifest["gaps"]["by_evidence_level"])

Total de gaps registrados: 4243
Por bucket estructural: {'2-9min': 1757, '10-70min': 1741, '70min-100h': 743, '>100h': 2}
Por evidence_level: {'structural_only': 3498, 'unconfirmed': 521, 'provisional_pattern_match': 224}


In [8]:
import json

print("Casos extraordinarios (>= umbral configurado):")
print(json.dumps(result.extraordinary_gaps, indent=2, ensure_ascii=False, default=str))

Casos extraordinarios (>= umbral configurado):
[
  {
    "gap_type_structural": "inter_contract",
    "source_file_left": "03_mnq_12_20.Last.txt",
    "source_file_right": "04_mnq_03_21.Last.txt",
    "contract_left": "Z20",
    "contract_right": "H21",
    "previous_timestamp": "2020-12-18 03:00:00",
    "next_timestamp": "2020-12-21 03:01:00",
    "duration_seconds": 259260.0,
    "structural_bucket": "70min-100h",
    "recurrence": 743,
    "provisional_interpretation_utc_hypothesis": "PROVISIONAL bajo hipótesis UTC: duración compatible con un cierre de fin de semana / feriado de mercado. No confirmado.",
    "evidence_level": "provisional_pattern_match"
  },
  {
    "gap_type_structural": "intra_file",
    "source_file_left": "04_mnq_03_21.Last.txt",
    "source_file_right": "04_mnq_03_21.Last.txt",
    "contract_left": "H21",
    "contract_right": "H21",
    "previous_timestamp": "2020-12-31 22:00:00",
    "next_timestamp": "2021-01-03 23:01:00",
    "duration_seconds": 262860.0,


## 5. Notas sobre zona horaria (sin resolver en S00)

- El indice se persiste **tz-naive**. No se aplico `tz_localize` ni
  `tz_convert`.
- `timezone_assumption` es una suposicion heredada (`UTC`), **no**
  confirmada documentalmente -- ver `timezone_evidence` en el summary.
- `timestamp_semantics` (inicio vs cierre de barra) queda explicitamente
  `unknown_not_confirmed`.
- La conversion a `America/New_York` y la aplicacion de calendario
  pertenecen a S01.

In [9]:
print(json.dumps(result.summary, indent=2, ensure_ascii=False, default=str))

{
  "name": "mnq_raw_v2",
  "pipeline_version": "s00_v2",
  "shape": [
    2172640,
    6
  ],
  "columns": [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "contract"
  ],
  "index_type": "DatetimeIndex",
  "index_tz": "tz-naive (sin zona horaria en el índice persistido)",
  "timezone_assumption": "UTC",
  "timezone_evidence": "inferred_not_confirmed",
  "timestamp_semantics": "unknown_not_confirmed",
  "bar_interval": "1_minute",
  "price_type": "Last",
  "price_type_evidence": "inferred_from_filename",
  "datetime_min": "2019-12-23T03:01:00",
  "datetime_max": "2026-04-17T20:18:00",
  "n_sources": 26,
  "gaps_summary": {
    "total_gaps": 4243,
    "by_structural_bucket": {
      "2-9min": 1757,
      "10-70min": 1741,
      "70min-100h": 743,
      ">100h": 2
    },
    "by_evidence_level": {
      "structural_only": 3498,
      "unconfirmed": 521,
      "provisional_pattern_match": 224
    },
    "extraordinary_cases": [
      {
        "gap_type_structural": "i